# ကလိပ် ၀၅ - အေးဂျန်ဆစ် RAG


## Setup

ဒီ notebook မှာ Microsoft Agent Framework ကို အသုံးပြုပြီး Agentic RAG (Retrieval-Augmented Generation) ပုံစံကို ပြသပေးထားပါတယ်။

**လိုအပ်ချက်များ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — သင့် Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — သင့် Azure AI Search API key
- Environment variables မှတဆင့် Azure OpenAI deployment ကို ပြင်ဆင်ထားရမည်
- Azure CLI မှာ authenticated ဖြစ်ရမည် (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG ဆိုတာဘာလဲ?

ရိုးရာ RAG သည် တည်ငြိမ်သော လုပ်ငန်းစဉ်တစ်ခုကို လိုက်နာသည်။ စာရွက်စာတမ်းများကို ရယူပြီးနောက် တုံ့ပြန်ချက် ထုတ်ပေးသည်။ **Agentic RAG** သည် အက်ဂျင့်ကို အချက်အလက်များကို **ဘယ်အချိန်** နှင့် **ဘယ်လို** ရယူရမယ်ဆိုတဲ့ ဆုံးဖြတ်ခွင့်ကို ပိုမိုပေးပြီး တိုးတက်စေသည်။

Agentic RAG ဖြင့် အက်ဂျင့်သည် -
- မေးခွန်းတစ်ခုကို ဖြေဆိုရန် မပြုမီ ရယူမှု လိုအပ်မလားကို **ဆုံးဖြတ်** နိုင်သည်
- မေးလေ့ရှိသည့် ဒေတာရင်းမြစ် သို့မဟုတ် ကိရိယာကို **ရွေးချယ်** နိုင်သည်
- ရယူထားသော ရလဒ်များကို **အကဲဖြတ်** ပြီး ပထမဆုံးကြိုးစားမှု မလုံလောက်ပါက နောက်ထပ် ရယူမှုများ ဆောင်ရွက်နိုင်သည်
- ရယူထားသည့် အချက်အလက်များကို အဆင့်အများစွာ ပေါင်းစပ်၍ တစ်ခုတည်းသော တုံ့ပြန်ချက်သို့ **ပေါင်းစည်း** နိုင်သည်

ဤသည်ကြောင့် အက်ဂျင့်သည် မတည်ငြိမ်သော ရယူပြီးနောက် ဖန်တီးမှု လုပ်ငန်းစဉ်ထက် ပိုမိုအဆင်ပြေပြီး တိကျမှန်ကန်မှုရှိစေသည်။


## ရှာဖွေရေးကိရိယာတစ်ခု ဖန်တီးခြင်း

Agentic RAG တွင် ပြင်ပဒေတာရင်းမြစ်များကို အေးဂျင့်က တောင်းဆိုလိုက်သလို ခေါ်နိုင်သော **ကိရိယာများ** အဖြစ် ထ.wrapထားသည်။ ၎င်းကြောင့် အေးဂျင့်သည် ရှာဖွေရေးကို မဖြစ်မနေ အဆင့်တစ်ခုအဖြစ်မဟုတ်ဘဲ လုပ်ဆောင်နိုင်သော အခြားလုပ်ဆောင်ချက်တစ်ခုအနေဖြင့် ဤအဆင့်ကို ကိုင်တွယ်နိုင်သည်။

အောက်တွင် ခရီးသွားအကြောင်း အချက်အလက်စနစ်တစ်ခုကို သတ်မှတ်ပြီး အဲဒီကိရိယာကို အေးဂျင့်က ရွှေ့ဆိုဒ်အချက်အလက်များကို ကြည့်ရှုနိုင်ဖို့ အမျိုးအစားအဖြစ် ထုတ်ဖော်ပြသထားသည်။


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG Agent တည်ဆောက်ခြင်း

ယခုတွင် အဖြေပြန်ရာတွင် **ဖော်ပြချက်မပေးမီ အမြဲသတင်းအချက်အလက် ရယူရန် အမိန့်ပြုထားသော agent** ကို ဖန်တီးပါမည်။ Agent သည် ပုဂ္ဂိုလ်ရေးလေ့ကျင့်မှုဒေတာပေါ် မူတည်ခြင်းမပြုဘဲ သိမ်းဆည်းထားသည့် အသိပညာရင်းမြစ်အပေါ် အခြေခံ၍ ဖြေကြားမှုများ ပေးနိုင်ရန် `search_travel_knowledge` ကိရိယာကို အသုံးပြုသည်။


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ပြန်လည် ရှာဖွေမှု — Maker-Checker ပုံစံ

Agentic RAG ၏ အဓိကအားသာချက် တစ်ခုမှာ **ပြန်လည် ရှာဖွေမှု** ဖြစ်သည်။ အဲဒီအေးဂျင့်သည် မိမိရရှိထားသော အချက်အလက်အချို့ကို အတည်ပြုရန်၊ ပိုမိုတိကျစေရန် သို့မဟုတ် ထပ်မံ တိုးချဲ့ရန် ရှာဖွေရေး အကြိမ်ရေများ ပြုလုပ်နိုင်သည် — "maker-checker" လိုင်းလုပ်ငန်းစဉ်နှင့် ဆင်တူသည်။

1. **Maker အဆင့်**: အေးဂျင့်သည် မူလအချက်အလက် ပေးဆောင်ပြီး တုံ့ပြန်ချက်ကို ကြိုးပမ်းရေးသားသည်။
2. **Checker အဆင့်**: အေးဂျင့်သည် အသေးစိတ်အချက်အလက်များကို အတည်ပြုရန် သို့မဟုတ် အချက်အလက်ရှင်းရှင်းလင်းလင်း မဖြည့်သွင်းသေးသောနေရာများအား ပြည့်စုံစေရန် ထပ်မံ ရှာဖွေမှုများ ပြုလုပ်သည်။

အောက်တွင် အေးဂျင့်အား ခရီးသွားရန်နေရာများစွာနှိုင်းယှဉ်စရာ လိုအပ်သည့် မေးခွန်းတစ်ခု မေးမြန်းထားပြီး၊ အစဉ်တစ်ကြိမ်ထက် ပို၍ ရှာဖွေရန် တိုက်တွန်းထားသည်။


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် သင်သည် Microsoft Agent Framework ကို အသုံးပြု၍ **Agentic RAG** စနစ်တစ်ခုကို တည်ဆောက်နည်းကို သင်ယူခဲ့ပါသည်-

- **Agentic RAG** သည် ကိုယ်ပိုင်ဆုံးဖြတ်ချက် ချနိုင်သော အေးဂျင့်များကို သတင်းအချက်အလက် ရယူခြင်း အချိန်ကို အလိုအလျောက်ဆုံးဖြတ်ခွင့် ပေးသည့်အတွက် ရယူမှုကို ပုံမှန် ညိှနိုင်းထားခြင်း မဟုတ်ဘဲ အပြောင်းအလဲရှိစေသည်။
- **ကိရိယာများအား ဒေတာရင်းမြစ်အဖြစ်**: အပြင်က သတင်းအချက်အလက် စုစည်းများ (Azure AI Search ကဲ့သို့) ကို အေးဂျင့်က အသုံးပြုနိုင်သော ကိရိယာများအဖြစ် ထုပ်ပိုးထားသည်။
- **အဆင့်လိုက် ရယူခြင်း**: Maker-checker ပုံစံသည် အေးဂျင့်အား လှည့်ကွက်လိုက် ရယူမှုများ ပြုလုပ်ခွင့်ပြု၍၊ ရှာဖွေ၊ အတည်ပြုနှင့် ညှိနှိုင်းမှ တစ်ဆင့် နောက်ဆုံးဖြေကြားချက် ထုတ်ပေးသည်။

ထုတ်လုပ်မှုတွင်၊ ကြီးမားသော ခရီးသွားစာရွက်စာတမ်း ရယူမှုအား ကိုင်တွယ်ရန် အတွက် သင်သည် in-memory `TRAVEL_KNOWLEDGE_BASE` ကို တကယ့် Azure AI Search အညွှန်းတစ်ခုဖြင့် အစားထိုးမည် ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
